# Literature Review Assistant with LLM

This notebook demonstrates how you can use UVA RC GenAI to assist with literature review tasks.

⚠️ **Important:** Always verify LLM outputs, especially citations. LLMs can generate plausible-sounding but fake references.

## Section 1: Setup

First, import libraries and configure your API connection.

In [ ]:
import requests
import json
from IPython.display import Markdown, display

API_URL = 'https://open-webui.rc.virginia.edu/api/chat/completions' 
API_KEY = 'sk-36d3a24c32d94bc4bdf4cf6beb90d1a8'
MODEL = "Kimi K2.5"

print("✅ Configuration loaded")

In [ ]:
def ask_llm(prompt, temperature=0.7, max_tokens=2000, show_stream=False):
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature,
        "max_tokens": max_tokens,
        "stream": True  # OpenWebUI uses streaming by default
    }

    full_text = ""

    with requests.post(API_URL, headers=headers, json=payload, timeout=120) as resp:
        resp.raise_for_status()

        for line in resp.iter_lines(decode_unicode=True):
            if not line:
                continue

            if line.startswith("data: "):
                data = line.removeprefix("data: ").strip()

                if data == "[DONE]":
                    break

                try:
                    chunk = json.loads(data)
                    delta = chunk["choices"][0].get("delta", {})
                    token = delta.get("content", "")

                    if token:
                        if show_stream:
                            print(token, end="", flush=True)
                        full_text += token
                except (json.JSONDecodeError, KeyError, IndexError):
                    continue

    return full_text

def display_response(response):
    # Display formatted LLM response
    display(Markdown(response))

# Test the connection
print("Testing API connection...")
display_response(ask_llm("Hello! Please respond with 'API connection successful.'"))

## Section 2: Single Paper Analysis

Let's analyze a single research paper. Replace this with any abstract.

In [ ]:
paper_abstract = """
Title: Pan-cancer spatial atlas of tertiary lymphoid structures

Abstract:
Tertiary lymphoid structures (TLSs) are critical regulators of antitumor immunity, yet their spatial 
organization, maturation, and clinical relevance remain incompletely defined across cancers. We analyzed spatial 
transcriptomics spanning 12 cancer types to construct a pan-cancer TLS atlas and characterized TLS spatial architecture 
and maturation states. TLS maturation was accompanied by coordinated remodeling of distinct niche cell populations and 
distance-dependent gradients in tumor programs, orthogonally supported by ultrahigh-plex single-cell spatial profiling. 
To enable scalable TLS profiling, we trained an artificial intelligence framework that predicts TLS maturation states 
directly from hematoxylin and eosin-stained images and evaluated it across TCGA and independent therapy cohorts. We 
further derived a maturation-aware composite score capturing intratumoral TLS state composition, which robustly 
stratifies patients across cancer and treatment contexts, outperforming conventional TLS metrics.
"""

print(f"Paper Loaded \nLength: {len(paper_abstract)} characters")

In [ ]:
# Task 1: Plain language summary
prompt = f"""
Breifly explain this research paper simply to a graduate student new to the field:

{paper_abstract}

Provide:
1. What the study did
2. The main finding
3. Why this matters
"""

display_response(ask_llm(prompt, temperature=0.3))

In [ ]:
# Task 2: Extract structured information
prompt = f"""
Extract key information from this abstract:

{paper_abstract}

Format:
## Research Question
## Methodology
## Key Findings
## Limitations
"""

display_response(ask_llm(prompt, temperature=0.3))

In [ ]:
# Task 3: Generate research questions
prompt = f"""
Based on this abstract, generate 3 novel research questions that extend this work:

{paper_abstract}

For each question:
1. The question
2. Why it's important
3. A feasible approach
"""

display_response(ask_llm(prompt, temperature=0.8))

## Section 3: Multiple Paper Analysis

In [ ]:
# Define list of abstracts
abstracts = [
    {
        "title": "Paper 1: Forest carbon protocols underestimate climate-driven carbon loss risks",
        "text": """Abstract: Although the reduction of fossil fuel emissions remains of the utmost importance to
mitigate climate change, maintaining and enhancing carbon sinks in forests have
been widely promoted as nature-based climate solutions1–4. However, disturbances
that could result in losses of forest carbon stocks are poorly accounted for when
estimating the potential role of forests in climate mitigation5–7. This makes it difficult
to appropriately size ‘buffer pools’: a mechanism designed to compensate for
unintended carbon losses in carbon crediting projects8,9. Here we use forest inventory,
satellite data, disturbance modelling and machine learning to map reversal (carbon
loss) risk in the contiguous United States (CONUS) from natural disturbance. Across
CONUS forests, we show that climate change increases the 100-year risk of carbon
losses from natural disturbance, particularly in California and the Intermountain West.
The current buffer pool of the largest CONUS forest climate mitigation programme is
likely too small by an average factor of 6.3, and this could range from 2.2- to 8.0-fold
too small when considering uncertainties around future climate scenarios, disturbance
severity and other carbon pools. We provide spatially explicit maps of the long-term
risks to forest carbon losses from natural disturbances, which highlight that current
methodologies used for constructing carbon offset buffer pools require revisions to
succeed under climate change."""
    },
    {
        "title": "Paper 2: Multistage assessment of construction delay factors using expert evaluation and real project data",
        "text": """Abstract: Construction delays remain a major challenge, especially in developing countries where financial,
administrative, and resource constraints intensify schedule disruptions. This study identifies and
prioritizes construction delay factors through a three-phase research framework consisting of a
literature review, an expert survey evaluation, and validation using real-life construction projects in
Egypt. First, a comprehensive review of global literature led to the identification of 98 delay factors,
which were classified into four main categories: owner-related factors (25 factors, 26%), contractor-
related factors (34 factors, 35%), consultant/design-related factors (14 factors, 14%), and external
factors (25 factors, 26%). Second, an expert survey was conducted to evaluate the previously identified
delay factors. This phase employed a composite scoring approach that integrates both the frequency
of occurrence and the impact of each factor to rank the most critical delay drivers. The findings
indicate that owner-related factors constitute the most significant sources of delay, accounting for
approximately 50% of the top 22 critical factors. Finally, the reliability of these findings was validated
using the top 22 factors and a dataset of 141 real construction projects, showing strong alignment
between the survey-based rankings and actual outcomes. Key factors, such as frequent change orders
and design modifications, remained top-ranked, while some factors ranked higher or lower in practice,
indicating minor variations in their relative importance under real project conditions. The study
contributes a validated dataset of delay factors derived from both expert evaluation and real project
evidence, providing a strong foundation for future predictive modeling applications using artificial
intelligence and machine learning."""
    },
    {
        "title": "Paper 3: Intelligent financial forecasting using transformers, neuro-symbolic AI, and agent-based systems",
        "text": """Abstract: Forecasting the stock market is a difficult task because of the
volatile price movements and complex temporal dependencies. Established
models frequently do not realize these unstable trends, which leads to
changeable forecasting. This paper introduces a comfortable AI-driven
framework that incorporates a sequence-to-prediction transformer model
with LLM-based decision-making for accurate and understandable stock
price prediction. The prediction starts with a transformer-enabled deep
learning approach, which forecasts closing prices for the NIFTY Consumer
Durables index based on multi-head attention mechanisms for trend identification. The predicted results are then provided to two decision-making
approaches: Neuro-Symbolic AI and an Advanced AI Agent Architecture, to
check for accuracy as well as interpretability in financial forecasting. The
NSAI model is focused on predictions that are derived from rule-based
reasoning, and thus, it ensures that they are in line with actual market
behaviors and risk factors. On the other hand, the design of Advanced AI
Agent Architecture aimed at enhancing the quality of decisions by utilizing
the LLM-based information, bringing in external financial information feeds
and memory-based historical data for the trading decision making process.
With these AI methods, the model can adapt to changes in market more
effectively and, thus, provide more accurate forecasts. This study goes a step
further in the development of financial markets analysis by combining deep
learning with symbolic reasoning, thereby verifying that the stock market
price predictions are not only correct but also interpretable and useable by
investors and traders. Through the use of AI-driven stock market strategies,
this research work provides a more flexible and explainable way of stock
market forecasts, thus it significantly advances the financial market analysis
which in turn can be utilized for a better investment method."""
    }
]

print(f"Loaded {len(abstracts)} papers for analysis")
for paper in abstracts:
    print(f"  - {paper['title']}")

# Analyze each abstract with progress tracking
results = []

for i, paper in enumerate(abstracts, 1):
    print(f"\n{'='*60}")
    print(f"Processing {i}/{len(abstracts)}: {paper['title']}")
    print('='*60)
    
    prompt = f"""Summarize this research paper in 2-3 bullet points:
    
Title: {paper['title']}
{paper['text']}

Format:
• Key finding 1
• Key finding 2  
• Key finding 3 (if applicable)
"""
    summary = ask_llm(prompt, temperature=0.3)
    results.append({
        "title": paper['title'],
        "summary": summary
    })
    print(summary)

print(f"\nFinished! Processed {len(results)} papers")    

## Tips
**Set API key** via environment variable: `export OPENWEBUI_API_KEY=...`

**Temperature guide**:
   - 0.1-0.3: Summarization, fact extraction
   - 0.7-0.9: Brainstorming, creative tasks

⚠️ **Always verify LLM-generated citations!**